# FlowGuard - Phase 3: Federated Learning

In [ ]:
import os, sys

REPO_DIR = "/content/flowguard"
if not os.path.exists(REPO_DIR):
    raise RuntimeError("Run notebook 00_setup_and_validate.ipynb first.")
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from src.utils.config import load_config, get_device
from src.utils.reproducibility import setup_reproducibility

config = load_config("configs/phase3_federated.yaml")
setup_reproducibility(config)
device = get_device(config)
print(f"Device: {device}")

In [ ]:
from src.training.federated import run_federated_simulation
history = run_federated_simulation("configs/phase3_federated.yaml")

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, metrics in history['client_metrics'].items():
    losses = [m['loss'] for m in metrics]
    accs = [m['accuracy'] for m in metrics]
    axes[0].plot(losses, label=name)
    axes[1].plot(accs, label=name)

axes[0].set_title('Client Loss per Round')
axes[0].legend()
axes[1].set_title('Client Accuracy per Round')
axes[1].legend()
plt.savefig('results/plots/phase3_convergence.png', dpi=100)
plt.show()

In [ ]:
import torch
from src.models.flowguard import FlowGuard
from src.evaluation.protocols import evaluate_protocol_b

model = FlowGuard(config)
model.load_state_dict(torch.load("checkpoints/phase3/final_global.pt", map_location=device))
model.to(device)

print("Protocol B - Federated Model:")
evaluate_protocol_b(model, config, device)